# Interactive Random-Wave Single-Chain Scattering

This notebook duplicates the line-network scattering workflow, but keeps only one vortex-line family, `S12 = {phi1 + i phi2 = 0}`. This is the "single chain / one complex field" test case.

The first run is monochromatic: `K_DISTRIBUTION = "single_shell"`. The sampled-chain scattering is compared with a Gaussian conditional-sampling calculation of the same monochromatic line correlation from `smpl/rw_line_scattering.py`. The Gaussian-sampling curve is scaled to the sampled-chain curve at one anchor point, so the comparison is primarily a shape check.


In [ ]:
import importlib
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pyvista as pv
import numpy as np

import rw_line_network as r
import rw_line_scattering as s

r = importlib.reload(r)
s = importlib.reload(s)

print("PyVista", pv.__version__)
print("Notebook imports/reloads complete")


## Parameters

The random-wave and rendering parameters are assigned on `r`; the scattering-specific parameters are assigned on `s`.

Two single-chain options matter for this notebook:

- `s.SCATTERING_AMPLITUDE_METHOD = "line_segments" | "points"`: `"line_segments"` is the continuous-line calculation and is the right default for testing the high-Q `pi d / Q` line asymptote. `"points"` is useful for bead-debugging but develops a high-Q self-term floor set by the point spacing.
- `s.POINT_WEIGHT_MODE = "unit" | "arclength"`: only affects `"points"`; `"unit"` keeps equal bead weights, while `"arclength"` makes point samples approximate a continuous line integral by assigning each line sample weight `LINE_SAMPLE_SPACING` times the family weight.
- `s.SCATTERING_Q_CHUNK_SIZE`: number of scalar Q values evaluated per progress chunk. It changes progress reporting and peak runtime granularity, not the scattering formula.


In [ ]:
# Random-wave sampling settings
r.GRID_SIZE = 32
r.NUM_BLOCK = 2
r.BLOCK_OVERLAP = 1
r.RANDOM_SEED = 894894
r.NUM_MODES = (128, 128, 128)
r.K_DISTRIBUTION = "single_shell"
r.K0 = (2, 2, 2)
r.r_SIGMA_K = (0.0, 0.0, 0.0)
r.r_K_MIN = (1.0, 1.0, 1.0)
r.r_K_MAX = (1.0, 1.0, 1.0)
r.SHARED_K_VECTORS = False

# phi3 is built by the original structure helper but ignored below.
r.COUPLE_PHI2_PHI3 = False
r.PHI23_COUPLING_C = 0.0

# Vortex tracing settings
r.USE_VORTEX_TRACING = True
r.VORTEX_FACE_PREFILTER = True
r.VORTEX_FACE_ZERO_TOL = 0.02
r.SMOOTH_VORTEX_LINES = True
r.VORTEX_SMOOTHING_SCALE = 2
r.VORTEX_TUBE_RADIUS = 0.25

# Crosslinks are not part of the single-chain test.
r.CROSSLINK_SEARCH_RADIUS = 1.25
r.CROSSLINK_MERGE_RADIUS = 0.5
r.CROSSLINK_BALL_RADIUS = 1.5

# Render preview settings
r.WINDOW_SIZE = (1000, 1000)
r.CAMERA_ZOOM = 0.82
r.CAMERA_AZIMUTH_DEGREES = 30.0
r.CAMERA_POLAR_DEGREES = 60.0
r.SHOW_BOUNDING_BOX = True
r.BOX_SIZE_L = None

# Scattering settings
s.LINE_SAMPLE_SPACING = 0.15
s.POINT_WEIGHT_MODE = "arclength"
s.S12_SCATTERING_WEIGHT = 1.0
s.S13_SCATTERING_WEIGHT = 0.0
s.SCATTERING_AMPLITUDE_METHOD = "points"  # "line_segments" or "points"
s.SCATTERING_WINDOW = "tukey_box"  # "none", "tukey_box", "hann_box", or "gaussian"
s.SCATTERING_WINDOW_TAPER = 0.05
s.SUBTRACT_WINDOWED_MEAN = True
s.WINDOW_MEAN_METHOD = "numeric_1d"
s.WINDOW_NORMALIZATION = "window_squared_volume"  # reduced analysis uses V2=int W^2 dV after raw scattering.
s.SUBTRACT_EXPLICIT_BOX_MEAN = False
s.ANALYTIC_MEAN_BUFFER_BLOCKS = 0
s.ANALYTIC_MEAN_BUFFER_MODE = "none"  # "none", "incoherent", or "coherent"
s.ANALYTIC_MEAN_BUFFER_NORMALIZE_TOTAL = False
s.DYNAMIC_LINE_SAMPLE_SPACING = False
s.DYNAMIC_LINE_SAMPLE_EXPONENT = 0.5
s.DYNAMIC_LINE_SAMPLE_POWER2_SUBSETS = True
s.INCLUDE_CROSSLINK_POINTS = False
s.CROSSLINK_SCATTERING_WEIGHT = 0.0

# Show the actual point scatterers used for I(Q) on the real-space preview.
s.SHOW_STRUCTURE_LINES = True
s.SHOW_SCATTERING_SAMPLE_POINTS = True
s.SCATTERING_SAMPLE_POINTS_AS_BALLS = True
s.SCATTERING_SAMPLE_BALL_RADIUS = 0.25
s.SCATTERING_SAMPLE_POINT_COLOR = "black"
s.SCATTERING_SAMPLE_POINT_SIZE = 2.0
s.SCATTERING_SAMPLE_POINT_OPACITY = 1.0

s.Q_MIN = 0.1
s.Q_MAX = 10
s.NUM_Q = 80
s.Q_SPACING = "log"
s.NUM_Q_DIRECTIONS = 96
s.SCATTERING_SEED = None
s.INTENSITY_NORMALIZATION = "none"  # raw windowed intensity; reduced units are applied below.
s.NORMALIZE_I0 = False
s.Q_AXIS_SCALE = "mean_k"  # "mean_k" or "raw"
s.NUM_SEED_AVERAGE = 8
s.STRUCTURE_SEED_START = r.RANDOM_SEED
s.STRUCTURE_SEED_STRIDE = 1009
s.SCATTERING_FLATTEN_Q_DIRECTIONS = False
s.SCATTERING_Q_CHUNK_SIZE = 10

# Gaussian conditional-sampling comparison.
MODEL_R_MIN = 1e-3
MODEL_R_MAX = 5e2
MODEL_NR = 5000
MODEL_N_SAMP = 2**15
MODEL_TAIL_START_FRACTION = 0.8

print({
    "sampling": {
        "GRID_SIZE": r.GRID_SIZE,
        "NUM_BLOCK": r.NUM_BLOCK,
        "BLOCK_OVERLAP": r.BLOCK_OVERLAP,
        "STITCHED_GRID_SIZE": r.expanded_grid_size(r.GRID_SIZE, r.NUM_BLOCK),
        "K_DISTRIBUTION": r.K_DISTRIBUTION,
        "K0": r.K0,
    },
    "scattering": {
        "LINE_SAMPLE_SPACING": s.LINE_SAMPLE_SPACING,
        "POINT_WEIGHT_MODE": s.POINT_WEIGHT_MODE,
        "SCATTERING_AMPLITUDE_METHOD": s.SCATTERING_AMPLITUDE_METHOD,
        "SCATTERING_WINDOW": s.SCATTERING_WINDOW,
        "SCATTERING_WINDOW_TAPER": s.SCATTERING_WINDOW_TAPER,
        "SUBTRACT_WINDOWED_MEAN": s.SUBTRACT_WINDOWED_MEAN,
        "WINDOW_NORMALIZATION": s.WINDOW_NORMALIZATION,
        "SHOW_STRUCTURE_LINES": s.SHOW_STRUCTURE_LINES,
        "SHOW_SCATTERING_SAMPLE_POINTS": s.SHOW_SCATTERING_SAMPLE_POINTS,
        "Q_MIN": s.Q_MIN,
        "Q_MAX": s.Q_MAX,
        "NUM_Q": s.NUM_Q,
        "NUM_Q_DIRECTIONS": s.NUM_Q_DIRECTIONS,
        "INTENSITY_NORMALIZATION": s.INTENSITY_NORMALIZATION,
        "Q_AXIS_SCALE": s.Q_AXIS_SCALE,
        "NUM_SEED_AVERAGE": s.NUM_SEED_AVERAGE,
        "STRUCTURE_SEEDS": s.structure_seed_values(),
        "SCATTERING_Q_CHUNK_SIZE": s.SCATTERING_Q_CHUNK_SIZE,
    },
    "gaussian_sampling_model": {
        "MODEL_R_MIN": MODEL_R_MIN,
        "MODEL_R_MAX": MODEL_R_MAX,
        "MODEL_NR": MODEL_NR,
        "MODEL_N_SAMP": MODEL_N_SAMP,
        "MODEL_TAIL_START_FRACTION": MODEL_TAIL_START_FRACTION,
    },
})


## Build Single-Chain Structure

The module helper still builds the full network first, then `s.as_single_s12_structure(...)` keeps only `S12` for the one-field/single-chain test.


In [ ]:
print("[build] building full line structure for preview/reference seed...")
t_build0 = time.perf_counter()
full_structure = s.build_line_structure(
    line_sample_spacing=s.LINE_SAMPLE_SPACING,
    include_crosslinks=False,
    print_timing=True,
    timing_label="preview seed",
)
print(f"[build] full structure finished in {time.perf_counter() - t_build0:.2f}s")

print("[build] filtering to single S12 chain...")
t_filter0 = time.perf_counter()
structure = s.as_single_s12_structure(full_structure, line_sample_spacing=s.LINE_SAMPLE_SPACING)
print(f"[build] S12 filter finished in {time.perf_counter() - t_filter0:.2f}s")

print("single-chain sampled scattering points:", len(structure.points))
print("single-chain point weight mode:", s.POINT_WEIGHT_MODE)
print("single-chain point weight value:", float(structure.weights[0]) if len(structure.weights) else 0.0)
print("single-chain continuous line segments:", len(structure.segment_starts))
print("S12 line points/lines:", structure.s12_lines.n_points, structure.s12_lines.n_lines)
print("S13 retained line points/lines:", structure.s13_lines.n_points, structure.s13_lines.n_lines)


## Real-Space Preview

In [ ]:
plotter = s.make_structure_plotter(structure)
plotter.show(jupyter_backend="static")


## Scattering Curve And Gaussian-Sampling Reference


In [ ]:
def status(message):
    print(message, flush=True)


def load_gaussian_sampling_model_module():
    import importlib.util
    import sys
    from pathlib import Path
    model_path = Path("smpl") / "rw_line_scattering.py"
    if not model_path.exists():
        model_path = Path.cwd() / "smpl" / "rw_line_scattering.py"
    status(f"[model] loading Gaussian-sampling module from {model_path}...")
    t0 = time.perf_counter()
    spec = importlib.util.spec_from_file_location("rw_line_scattering_smpl", model_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    status(f"[model] module loaded in {time.perf_counter() - t0:.2f}s")
    return module


def gaussian_sampling_single_chain_reference(q_over_k):
    status(
        f"[model] computing Gaussian-sampling reference: "
        f"Nr={MODEL_NR}, N_samp={MODEL_N_SAMP}, Q={len(q_over_k)}"
    )
    t0 = time.perf_counter()
    ris = load_gaussian_sampling_model_module()
    r_model = np.linspace(MODEL_R_MIN, MODEL_R_MAX, MODEL_NR)
    k_radii = np.ones(1, dtype=float)
    k_weights = np.ones(1, dtype=float)
    t_cl0 = time.perf_counter()
    M_J, C_L = ris.compute_CL_general(
        r_model,
        k_radii,
        MODEL_N_SAMP,
        k_weights=k_weights,
        use_qmc=True,
        random_seed=r.RANDOM_SEED,
        progress=True,
    )
    status(f"[model] C_L sampling finished in {time.perf_counter() - t_cl0:.2f}s")
    C_L_coherent = ris.coherent_CL_general(C_L, k_radii, k_weights=k_weights)
    W_tail = ris.tail_window(r_model, MODEL_TAIL_START_FRACTION*MODEL_R_MAX, MODEL_R_MAX)
    t_h0 = time.perf_counter()
    I_model = ris.hankel_transform(r_model, C_L_coherent * W_tail, q_over_k)
    status(f"[model] Hankel transform finished in {time.perf_counter() - t_h0:.2f}s")
    status(f"[model] Gaussian-sampling reference complete in {time.perf_counter() - t0:.2f}s")
    return I_model, {"r_model": r_model, "M_J": M_J, "C_L": C_L, "C_L_coherent": C_L_coherent, "W_tail": W_tail}


t_cell0 = time.perf_counter()
status("[main] starting scattering and comparison cell")
seeds = s.structure_seed_values()
q, iq_raw, iq_raw_std, iq_box, point_counts, window_diagnostics = s.compute_seed_averaged_single_chain_scattering(
    seeds,
    print_timing=True,
)
status("[main] computing Q-axis metadata and reduced units...")
k_mean = float(np.mean(r._field_parameter_values(r.K0, "K0")))
wave_coordinate_scale = float(r.GRID_SIZE)
box_grid_side = float(r.GRID_SIZE - 1)
k_wave_angular_mean = 2.0 * np.pi * k_mean
k_angular_grid = k_wave_angular_mean / wave_coordinate_scale
box_l = r.BOX_SIZE_L if r.BOX_SIZE_L is not None else float(r.expanded_grid_size(r.GRID_SIZE, r.NUM_BLOCK) - 1)
box_volume_grid = box_l**3
window_volume = s.window_volume()
window_squared_volume = s.window_squared_volume()
preview_window = s.single_chain_window_diagnostics(structure)
line_measure_w_values = np.asarray([diag["linear_measure_w"] for diag in window_diagnostics], dtype=float)
line_measure_w2_values = np.asarray([diag["linear_measure_w2"] for diag in window_diagnostics], dtype=float)
d_from_w_values = np.asarray([diag["density_from_w"] for diag in window_diagnostics], dtype=float)
d_from_w2_values = np.asarray([diag["density_from_w2"] for diag in window_diagnostics], dtype=float)
d_theory = k_angular_grid**2 / (3.0*np.pi)
d_measured_w = float(np.nanmean(d_from_w_values))
d_measured_w2 = float(np.nanmean(d_from_w2_values))
d = d_measured_w2 if np.isfinite(d_measured_w2) and d_measured_w2 > 0.0 else d_theory
d_source = "measured window-squared line density" if d == d_measured_w2 else "theoretical K^2/(3*pi) fallback"
N = d**2
line_density_est = d
line_length_est = d * box_volume_grid
q_grid_units = q
q_over_k = q / k_angular_grid
q_over_k_angular = q_over_k
q_scaled = q_over_k
q_axis_label = r"$Q/k$"

I_windowed = iq_raw / window_squared_volume
I_windowed_std = iq_raw_std / window_squared_volume
I_reduced = I_windowed / N
I_reduced_std = I_windowed_std / N
line_asymptote_reduced = np.pi / np.maximum(q_grid_units * d, np.finfo(float).tiny)
line_ratio_reduced = I_reduced / line_asymptote_reduced
line_ratio_reduced_std = I_reduced_std / line_asymptote_reduced

I_gaussian_raw, gaussian_model = gaussian_sampling_single_chain_reference(q_over_k)
status("[main] converting Gaussian-sampling reference to reduced intensity...")
model_line_density_k1 = 1.0 / (3.0*np.pi)
d_gaussian_theory = k_angular_grid**2 * model_line_density_k1
I_gaussian_windowed = k_angular_grid * I_gaussian_raw * (d / d_gaussian_theory)
I_gaussian_reduced = I_gaussian_windowed / N
line_ratio_gaussian_reduced = I_gaussian_reduced / line_asymptote_reduced
I_gaussian_raw_line_asymptote = np.pi * model_line_density_k1 / np.maximum(q_over_k, np.finfo(float).tiny)
I_gaussian_raw_scaled_by_model_asymptote = I_gaussian_raw / I_gaussian_raw_line_asymptote

# Backward-compatible names for saved outputs and older plotting expectations.
iq = iq_raw
I_gaussian_scaled = I_gaussian_reduced
I_scaled_by_line_asymptote = line_ratio_reduced
I_gaussian_scaled_by_line_asymptote = line_ratio_gaussian_reduced
# Backward-compatible aliases from the previous label convention.
porod_reduced = line_ratio_reduced
porod_reduced_std = line_ratio_reduced_std
porod_gaussian_reduced = line_ratio_gaussian_reduced
line_asymptote_for_settings = line_asymptote_reduced
line_asymptote_i0 = line_asymptote_reduced
gaussian_scale = np.nan
gaussian_normalization_scale = np.nan
line_measure_for_scaling = float(np.mean(line_measure_w_values))
normalization_factor_for_scaling = N
line_asymptote_prefactor = np.pi/d

status("reduced normalization calculation:")
status(f"  K = 2*pi*<k_cycles>/GRID_SIZE = {k_angular_grid:.6g}")
status(f"  theoretical line density d_theory = K^2/(3*pi) = {d_theory:.6g}")
status(f"  measured W density = {d_measured_w:.6g}")
status(f"  measured W^2 density = {d_measured_w2:.6g}")
status(f"  line density d used for scaling = {d:.6g} ({d_source})")
status(f"  N = d^2 = {N:.6g}")
status(f"  window-adjusted intensity I(Q) = I_raw(Q)/int(W^2 dV)")
status(f"  reduced intensity = I(Q)/d^2")
status(f"  left-panel local-line guide: I(Q)/d^2 = pi/(Q*d)")
status(f"  right-panel ratio: [I(Q)/d^2] / [pi/(Q*d)] should approach 1")
status("window diagnostics:")
status(f"  int W dV = {window_volume:.6g}")
status(f"  int W^2 dV = {window_squared_volume:.6g}")
status(f"  mean sum(line_weight*W) = {np.mean(line_measure_w_values):.6g}")
status(f"  mean sum(line_weight*W^2) = {np.mean(line_measure_w2_values):.6g}")
status(f"  mean density from W integral = {np.nanmean(d_from_w_values):.6g}")
status(f"  mean density from W^2 integral = {np.nanmean(d_from_w2_values):.6g}")
status(f"estimated line length d*V_box = {line_length_est:.6g}")
status(f"measured preview geometric S12 line length = {preview_window['geometric_length']:.6g}")
status(f"[main] scattering/comparison cell complete in {time.perf_counter() - t_cell0:.2f}s")


In [ ]:
print("[plot] rendering scattering comparison plot...")
t_plot0 = time.perf_counter()
fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.2), constrained_layout=True)
ax = axes[0]

ax.plot(q_scaled, I_reduced, color="black", lw=1.6, label="Trajectory")
if len(seeds) > 1:
    ax.fill_between(q_scaled, I_reduced - I_reduced_std, I_reduced + I_reduced_std, color="black", alpha=0.18, linewidth=0)

ax.plot(q_over_k, I_gaussian_reduced, color="tab:orange", lw=1.5, ls="--", label="Gaussian sampling")
ax.plot(q_scaled, line_asymptote_reduced, color="tab:blue", lw=1.2, ls=":", label=r"$\pi/(Qd)$")

ax.set_xlabel(q_axis_label)
ax.set_ylabel(r"$I(Q)/d^2$")
ax.set_xscale("log")
ax.set_yscale("log")

q_half_box = 4.0 * np.pi / box_l
q_half_box_scaled = q_half_box / k_angular_grid
ax.axvline(q_half_box_scaled, color="tab:red", lw=1.2, ls="--", alpha=0.8, label=r"$2\pi/(L/2)$")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)

ax = axes[1]
ax.plot(q_scaled, line_ratio_reduced, color="black", lw=1.6, label=r"Trajectory")
if len(seeds) > 1:
    ax.fill_between(
        q_scaled,
        line_ratio_reduced - line_ratio_reduced_std,
        line_ratio_reduced + line_ratio_reduced_std,
        color="black",
        alpha=0.18,
        linewidth=0,
    )
ax.plot(q_over_k, line_ratio_gaussian_reduced, color="tab:orange", lw=1.5, ls="--", label="Gaussian sampling")
ax.axhline(1.0, color="tab:blue", lw=1.2, ls=":")
ax.axvline(q_half_box_scaled, color="tab:red", lw=1.2, ls="--", alpha=0.8)
ax.set_xlabel(q_axis_label)
ax.set_ylabel(r"$QI(Q)/(\pi d)$")
ax.set_xscale("log")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)

print("half-box marker Q/k:", q_half_box_scaled)
print("seeds:", seeds)
print("single-chain scattering point counts:", point_counts)
print("mean random-wave k in cycles per grid:", k_mean)
print("K angular per grid unit:", k_angular_grid)
print("theoretical line density d_theory = K^2/(3*pi):", d_theory)
print("measured W density:", d_measured_w)
print("measured W^2 density:", d_measured_w2)
print("line density d used for scaling:", d, d_source)
print("N = d^2:", N)
print("active scattering box side L_box:", box_l)
print("box volume in grid units:", box_volume_grid)
print("int W dV:", window_volume)
print("int W^2 dV:", window_squared_volume)
print("mean sum(line_weight*W):", float(np.mean(line_measure_w_values)))
print("mean sum(line_weight*W^2):", float(np.mean(line_measure_w2_values)))
print("mean density from W integral:", float(np.nanmean(d_from_w_values)))
print("mean density from W^2 integral:", float(np.nanmean(d_from_w2_values)))
print("estimated line length d*V_box:", line_length_est)
print("raw I(Q) range:", float(iq_raw.min()), float(iq_raw.max()))
print("scattering amplitude method:", s.SCATTERING_AMPLITUDE_METHOD)
print("reduced I/d^2 range:", float(I_reduced.min()), float(I_reduced.max()))
print("line-guide ratio range:", float(line_ratio_reduced.min()), float(line_ratio_reduced.max()))
print(f"[plot] plot cell complete in {time.perf_counter() - t_plot0:.2f}s")


## Save Data

In [ ]:
print("[save] saving scattering outputs...")
t_save0 = time.perf_counter()
out_path = s.save_scattering_curve(
    "rw_single_chain_scattering_iq.csv",
    q,
    iq_raw,
    std=iq_raw_std,
    box_intensity=iq_box,
)
print(out_path)

np.savez_compressed(
    "rw_single_chain_scattering_compare.npz",
    q=q,
    q_over_k=q_over_k,
    q_scaled=q_scaled,
    iq_raw=iq_raw,
    iq_raw_std=iq_raw_std,
    I_windowed=I_windowed,
    I_windowed_std=I_windowed_std,
    I_reduced=I_reduced,
    I_reduced_std=I_reduced_std,
    line_ratio_reduced=line_ratio_reduced,
    line_ratio_reduced_std=line_ratio_reduced_std,
    porod_reduced=porod_reduced,
    porod_reduced_std=porod_reduced_std,
    q_over_k_angular=q_over_k_angular,
    wave_coordinate_scale=wave_coordinate_scale,
    box_grid_side=box_grid_side,
    box_l=box_l,
    I_gaussian_raw=I_gaussian_raw,
    I_gaussian_windowed=I_gaussian_windowed,
    I_gaussian_reduced=I_gaussian_reduced,
    line_ratio_gaussian_reduced=line_ratio_gaussian_reduced,
    porod_gaussian_reduced=porod_gaussian_reduced,
    model_line_density_k1=model_line_density_k1,
    k_angular_grid=k_angular_grid,
    d=d,
    d_source=d_source,
    d_theory=d_theory,
    d_measured_w=d_measured_w,
    d_measured_w2=d_measured_w2,
    d_gaussian_theory=d_gaussian_theory,
    N=N,
    line_density_est=line_density_est,
    line_length_est=line_length_est,
    line_measure_w_values=line_measure_w_values,
    line_measure_w2_values=line_measure_w2_values,
    d_from_w_values=d_from_w_values,
    d_from_w2_values=d_from_w2_values,
    window_volume=window_volume,
    window_squared_volume=window_squared_volume,
    line_asymptote_reduced=line_asymptote_reduced,
    I_gaussian_raw_line_asymptote=I_gaussian_raw_line_asymptote,
    I_gaussian_raw_scaled_by_model_asymptote=I_gaussian_raw_scaled_by_model_asymptote,
    box_volume_grid=box_volume_grid,
    point_counts=np.asarray(point_counts),
    seeds=np.asarray(seeds),
)
print("rw_single_chain_scattering_compare.npz")
print(f"[save] save cell complete in {time.perf_counter() - t_save0:.2f}s")
